In [1]:
import numpy as np
import pandas as pd
from time import time
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

import warnings
warnings.filterwarnings('ignore')

from dirarte.datasets import FicoDataset, CreditDataset, GermanDataset, CompasDataset, BailDataset
from dirarte import FeatureTweakingExplainer, OptimalActionExplainer, FocusExplainer

In [2]:
def run(
    dataset, 
    estimator='XGBoost', 
    cost_type='std',
    n_iter=5,
    max_samples=20,
):
    
    np.random.seed(0)
    results = {
        'dataset': [],
        'estimator': [],
        'cost_type': [],
        'n_samples': [],
        'method': [],
        'validity': [],
        'cost': [],
        'plausibility': [],
        'sparsity': [],
        'time': [],     
        'time per samples': [],   
    }
    
    for i in tqdm(range(n_iter)):
        
        X_train, X_test, y_train, y_test = dataset.get_dataset(split=True, test_size=0.2)
        constraints = dataset.constraints

        if estimator == 'XGBoost':
            clf = XGBClassifier(n_estimators=100, max_depth=6).fit(X_train, y_train)
        else:
            clf = RandomForestClassifier(n_estimators=100, max_depth=6).fit(X_train, y_train)
        if max_samples > 0:
            X_target = X_test[clf.predict(X_test) == 0][:max_samples]
        else:
            X_target = X_test[clf.predict(X_test) == 0]
            
        for method in ['FT', 'OAE', 'FOCUS (0.001)', 'FOCUS (0.01)', 'FOCUS (0.1)', 'FT++']:
            
            if method == 'FT':
                explainer = FeatureTweakingExplainer(clf, constraints, cost_type=cost_type, merge_level=1)
            elif method == 'OAE':
                explainer = OptimalActionExplainer(clf, constraints, cost_type=cost_type, time_limit=180)
            elif method == 'FOCUS (0.001)':
                explainer = FocusExplainer(clf, constraints, cost_type=cost_type, beta=0.001)
            elif method == 'FOCUS (0.01)':
                explainer = FocusExplainer(clf, constraints, cost_type=cost_type, beta=0.01)
            elif method == 'FOCUS (0.1)':
                explainer = FocusExplainer(clf, constraints, cost_type=cost_type, beta=0.1)
            elif method == 'FT++':
                explainer = FeatureTweakingExplainer(clf, constraints, cost_type=cost_type, merge_level=2)

            explainer = explainer.initialize(X_train)
            time_method = time()
            recourse = explainer.explain_recourse(X_target)
            time_method = time() - time_method            
            
            X_cf = recourse.counterfactual
            results['dataset'].append(dataset.name)
            results['estimator'].append(estimator)
            results['cost_type'].append(cost_type)
            results['n_samples'].append(len(X_target))
            results['method'].append(method)
            results['validity'].append(clf.predict(X_cf).mean())
            results['cost'].append(recourse.get_average_cost())
            results['plausibility'].append(recourse.get_average_plausibility())
            results['sparsity'].append(recourse.get_average_sparsity())
            results['time'].append(time_method)
            results['time per samples'].append(time_method / len(X_target))
                
    return pd.DataFrame(results)

In [ ]:
results_xg = []
for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
    result = run(
        dataset, 
        estimator='XGBoost', 
        cost_type='std',
        n_iter=5,
        max_samples=20,
    )
    results_xg.append(result)
results_xg = pd.concat(results_xg, ignore_index=True)

100%|██████████| 5/5 [2:20:35<00:00, 1687.03s/it]  


In [4]:
results_rf = []
for dataset in [FicoDataset(), CreditDataset(), GermanDataset(), CompasDataset(), BailDataset()]:
    result = run(
        dataset, 
        estimator='RandomForest', 
        cost_type='std',
        n_iter=5,
        max_samples=20,
    )
    results_rf.append(result)
results_rf = pd.concat(results_rf, ignore_index=True)

100%|██████████| 5/5 [4:04:54<00:00, 2938.83s/it]  


In [5]:
results = pd.concat([results_xg, results_rf], ignore_index=True)
results.to_csv('./results/results_baseline.csv', index=False)